# Murphy 1 · Chapter 7 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M7). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 7.1 · Power iteration finds the top eigenvector  
*Section §7.4 · code*

For A = [[4, 1], [2, 3]], start with a random vector and repeat v ← Av / ‖Av‖ fifty times. Estimate the top eigenvalue with vᵀAv / vᵀv and compare with np.linalg.eig.

<details><summary><b>Hint</b></summary>

Each multiplication grows the top eigenvector's share by λ₁/λ₂.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
A = np.array([[4., 1.], [2., 3.]])
v = np.random.default_rng(9).normal(size=2)
for _ in range(50):
    v = A @ v
    v = ...  # TODO: keep it unit length
lam = ...  # TODO: Rayleigh quotient
print("power iteration:", v.round(4), round(lam, 4))
print("numpy eig      :", np.linalg.eig(A)[0])

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
A = np.array([[4., 1.], [2., 3.]])
v = np.random.default_rng(9).normal(size=2)
for _ in range(50):
    v = A @ v
    v = v / np.linalg.norm(v)  # keep it unit length
lam = v @ A @ v / (v @ v)  # Rayleigh quotient
print("power iteration:", v.round(4), round(lam, 4))
print("numpy eig      :", np.linalg.eig(A)[0])

<details><summary><b>Answer and explanation</b></summary>

Eigenvalues are 5 and 2. Every multiply scales the λ = 5 direction 2.5× faster than the λ = 2 one, so after a few dozen steps only the top direction is left: v ≈ [0.707, 0.707], estimate ≈ 5. Station 7's PCA uses this method.

Expected output:
```
power iteration: [-0.7071 -0.7071] 5.0
numpy eig      : [5.+0.j 2.+0.j]
```

</details>

## 7.2 · Compress an image with truncated SVD  
*Section §7.5 · code*

Build a 64×64 'image' (sum of a few smooth patterns plus noise). Compute its SVD and reconstruct it from the top k singular values for k = 1, 2, 5, 10, 20. Print the relative error and the fraction of numbers stored.

<details><summary><b>Hint</b></summary>

U[:, :k] * s[:k] @ Vt[:k] rebuilds the rank-k approximation. Storage = k(64 + 64 + 1).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(10)
u = np.linspace(0, 1, 64)
img = np.outer(np.sin(3*u), np.cos(2*u)) + 0.5*np.outer(u, u**2) + 0.3*np.outer(np.cos(7*u), np.sin(5*u)) + 0.02*rng.normal(size=(64, 64))
U, s, Vt = np.linalg.svd(img)
for k in [1, 2, 5, 10, 20]:
    approx = ...  # TODO: keep the k strongest directions
    err = np.linalg.norm(img - approx) / np.linalg.norm(img)
    print(f"k={k:2d}  relative error {err:.4f}   stored {k*(64+64+1)/64**2:.1%}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(10)
u = np.linspace(0, 1, 64)
img = np.outer(np.sin(3*u), np.cos(2*u)) + 0.5*np.outer(u, u**2) + 0.3*np.outer(np.cos(7*u), np.sin(5*u)) + 0.02*rng.normal(size=(64, 64))
U, s, Vt = np.linalg.svd(img)
for k in [1, 2, 5, 10, 20]:
    approx = U[:, :k] * s[:k] @ Vt[:k]  # keep the k strongest directions
    err = np.linalg.norm(img - approx) / np.linalg.norm(img)
    print(f"k={k:2d}  relative error {err:.4f}   stored {k*(64+64+1)/64**2:.1%}")

<details><summary><b>Answer and explanation</b></summary>

The error drops fast for the first few k (the smooth structure) and then flattens at the noise level. Rank 5 stores about 16% of the numbers with a small error: truncated SVD keeps the strongest directions, which is the same idea as PCA (Murphy §7.5.5).

Expected output:
```
k= 1  relative error 0.2178   stored 3.1%
k= 2  relative error 0.0867   stored 6.3%
k= 5  relative error 0.0371   stored 15.7%
k=10  relative error 0.0319   stored 31.5%
k=20  relative error 0.0227   stored 63.0%
```

</details>

## 7.3 · Gradient of the squared error  
*Section §7.8 · derive*

Show that the gradient of f(w) = ‖Xw − y‖² is 2Xᵀ(Xw − y), and set it to zero to get the normal equations.

<details><summary><b>Hint</b></summary>

Write f = (Xw − y)ᵀ(Xw − y) = wᵀXᵀXw − 2yᵀXw + yᵀy, then use ∇(wᵀAw) = (A + Aᵀ)w and ∇(bᵀw) = b.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

∇f = (XᵀX + (XᵀX)ᵀ)w − 2Xᵀy = 2XᵀXw − 2Xᵀy = 2Xᵀ(Xw − y). Setting it to 0: XᵀXw = Xᵀy, so ŵ = (XᵀX)⁻¹Xᵀy when XᵀX is invertible. Geometrically, the residual Xw − y must be perpendicular to every column of X (Murphy §7.7.3, §11.2.2).

</details>

## 7.4 · Three ways to solve least squares  
*Section §7.7 · code*

With X random 100×3 and y = X[1, −2, 0.5] + noise, solve for w with (a) the normal equations, (b) np.linalg.lstsq, (c) the pseudo-inverse. Then make two columns nearly identical and compare the condition number and the answers.

<details><summary><b>Hint</b></summary>

np.linalg.cond(X.T @ X) measures how close to singular the system is.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(11)
X = rng.normal(size=(100, 3)); w_true = np.array([1., -2., 0.5])
y = X @ w_true + 0.1 * rng.normal(size=100)
for name, M in [("good", X), ("collinear", np.column_stack([X[:, 0], X[:, 0] + 1e-6 * rng.normal(size=100), X[:, 2]]))]:
    a = ...  # TODO: normal equations
    b = np.linalg.lstsq(M, y, rcond=None)[0]
    c = np.linalg.pinv(M) @ y
    print(f"{name:9s} cond(XtX)={np.linalg.cond(M.T @ M):.2e}")
    print("  normal", a.round(3), " lstsq", b.round(3), " pinv", c.round(3))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(11)
X = rng.normal(size=(100, 3)); w_true = np.array([1., -2., 0.5])
y = X @ w_true + 0.1 * rng.normal(size=100)
for name, M in [("good", X), ("collinear", np.column_stack([X[:, 0], X[:, 0] + 1e-6 * rng.normal(size=100), X[:, 2]]))]:
    a = np.linalg.solve(M.T @ M, M.T @ y)  # normal equations
    b = np.linalg.lstsq(M, y, rcond=None)[0]
    c = np.linalg.pinv(M) @ y
    print(f"{name:9s} cond(XtX)={np.linalg.cond(M.T @ M):.2e}")
    print("  normal", a.round(3), " lstsq", b.round(3), " pinv", c.round(3))

<details><summary><b>Answer and explanation</b></summary>

On well-conditioned data all three agree to 3 decimals. With two nearly identical columns the problem itself is ill-posed: the data only pins down the sum of their two weights (≈ 1), so every method returns huge weights of opposite sign that cancel. The normal equations also disagree with the others from the 4th digit on, because forming XᵀX squares the condition number and throws away precision. The real fix is to change the problem: ridge (a small λ makes the weights small and stable) or dropping one of the duplicate columns.

Expected output:
```
good      cond(XtX)=1.63e+00
  normal [ 0.997 -2.     0.51 ]  lstsq [ 0.997 -2.     0.51 ]  pinv [ 0.997 -2.     0.51 ]
collinear cond(XtX)=3.39e+12
  normal [ 3.64248499e+05 -3.64247366e+05  5.24000000e-01]  lstsq [ 3.64037649e+05 -3.64036516e+05  5.24000000e-01]  pinv [ 3.64037649e+05 -3.64036516e+05  5.24000000e-01]
```

</details>